# Classifier-free guidance in SiT: new interventions

At w = 4, guidance makes SiT's images recognizable but uniform, and the original study found most of the uniformity
comes from the first 30% of sampling, where guidance gives every sample of a class the same push. This notebook tests
new ways to handle that early push:

1. **Guidance interval:** switch guidance off at the start and on partway through. When should it start?
2. **Shuffled push:** keep the early push just as strong, but turn it to a different direction for each sample.
3. **Limited turn:** like the shuffled push, but never turn the push further than a set angle. Does that keep the
   extra variety and remove the recoloring the shuffled push caused?

**Before you start:** Runtime → Change runtime type → **T4 GPU**, and have `cfg-project-colab.zip` ready to upload
(or put it in the top level of your Google Drive).

**How to use it:** run **Setup** once per session, then either experiment. Each experiment is self-contained. The
**Reference runs** at the end reproduce the original study's results and are optional; if experiment 2 of the
reference runs has been done, both experiments reuse its baselines and compare against its results.

| Part | What it does | Time on a T4 (estimate) | Output |
|---|---|---|---|
| Setup | GPU check, code, packages, weights | ~5 min | 1.8 GB weights |
| Experiment 1 | Guidance interval: start times t = 0.1 to 0.5 | ~1–1.5 h | ~100 MB |
| Experiment 2 | Shuffled push vs score damping | ~45–75 min | ~60 MB |
| Experiment 3 | Limited turn: 30°, 60°, 90°, 120° (needs experiment 2) | ~50 min | ~50 MB |
| Reference R1 | The guidance vector over time (also a smoke test) | ~5–10 min | ~160 MB |
| Reference R2 | The original 13 interventions | ~2–4 h | ~250 MB |
| Reference R3 | Full logging of w = 4 vs w = 1, 16 seeds (off by default) | ~1 h + embedding | ~19 GB |

**Results will not match the repository's hashes bit for bit.** The starting noise is made on the CPU, so it is
identical on every device, but CUDA and Apple's GPU round differently, so final latents differ slightly. Treat these
runs as a replication: the conclusions should hold, the SHA-256 values in `metadata.json` will not.

# Setup

Run these cells once per session.

## S1. Check the GPU and choose where outputs go

Colab's local disk is wiped when the session ends, and free sessions can disconnect. Saving to Google Drive keeps
results across sessions, and every run skips batches that are already finished, so rerunning a cell after a
disconnect picks up where it stopped. Free Drive has 15 GB: enough for everything except reference run R3.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

USE_DRIVE = True  # set to False to keep everything on the session's local disk

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cfg-project-outputs"
else:
    OUT = "/content/cfg-project/outputs/colab"
print("outputs go to", OUT)

## S2. Get the code

The repository is private, so the code comes from a zip instead of a clone: `cfg-project-colab.zip` holds `bench/`,
the two documents and the intervention direction tables. Either upload it when the cell asks, or put it in the top
level of your Google Drive and it will be picked up from there.

SiT itself is public, so it is cloned from upstream and pinned to commit `cbde832`, the same commit as the
repository's submodule.

In [ ]:
import os, shutil
ZIP_ON_DRIVE = "/content/drive/MyDrive/cfg-project-colab.zip"

if not os.path.exists("/content/cfg-project/bench"):
    if USE_DRIVE and os.path.exists(ZIP_ON_DRIVE):
        zip_path = ZIP_ON_DRIVE
    else:
        from google.colab import files
        zip_path = "/content/" + next(iter(files.upload()))  # choose cfg-project-colab.zip
    !unzip -q -o "{zip_path}" -d /content

%cd /content/cfg-project
if not os.path.exists("sit/models.py"):
    !git clone -q https://github.com/willisma/sit sit
    !git -C sit checkout -q cbde832
!git -C sit rev-parse --short HEAD

## S3. Install packages

Colab already has torch, torchvision, numpy, scipy, pillow and matplotlib. These are the rest.

In [ ]:
!pip install -q torchdiffeq timm diffusers safetensors
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> T4 GPU"

## S4. Download the SiT-L/2 weights

Community weights from BiliSakura/SiT-diffusers, pinned to one revision. The checksum must match the one in the README.

In [ ]:
W = "pretrained_models/SiT-L-2-256_BiliSakura_2f02393/transformer.safetensors"
SRC = "https://huggingface.co/BiliSakura/SiT-diffusers/resolve/2f02393a3079dfa6210c10302cc927f59d2cb142/SiT-L-2-256/transformer/diffusion_pytorch_model.safetensors"
EXPECTED = "fd3c19ec1a7675f5648f5200acf4f1a94ae6a44ff5ff81a1011de8242d12edd6"

os.makedirs(os.path.dirname(W), exist_ok=True)
if not os.path.exists(W):
    !curl -fL -o {W} "{SRC}"
got = !sha256sum {W}
got = got[0].split()[0]
assert got == EXPECTED, f"checksum mismatch: {got}"
print("weights OK")

## S5. Shared settings

Paths and helpers every experiment uses.

In [ ]:
import csv, glob
from IPython.display import Image, display
import matplotlib.pyplot as plt
sys.path.insert(0, "bench")  # so cells can import the project's helpers (rotation checks, colour measure)

# Direction tables (each class's 3 shared guidance directions), fit on seeds 0-15. Ships in the zip.
T = "outputs/guidance_delta/SiT-L_2_16seeds/analysis/intervention_tables.npz"
# Held-out evaluation seeds (the scripts refuse seeds that overlap the fitting seeds).
TEST_SEEDS = list(range(16, 32))
TEST_SEED_ARG = " ".join(map(str, TEST_SEEDS))

# One output folder per experiment.
EXP1 = f"{OUT}/exp1_guidance_interval"
EXP2 = f"{OUT}/exp2_shuffled_push"
EXP3 = f"{OUT}/exp3_limited_turn"
REF1 = f"{OUT}/ref1_guidance_vector"
REF2 = f"{OUT}/ref2_interventions"
REF3 = "/content/cfg-project/outputs/colab/ref3_16seeds"  # local disk: too big for free Drive

def have_all(folder, seeds=TEST_SEEDS):
    """True when a condition folder has every seed's finished file."""
    return all(os.path.exists(f"{folder}/seed_{s}.npz") for s in seeds)

def reuse_from_ref2(run, conditions):
    """Copy finished conditions from reference run R2 into run, so they are not sampled twice."""
    for c in conditions:
        if have_all(f"{REF2}/{c}") and not have_all(f"{run}/{c}"):
            shutil.copytree(f"{REF2}/{c}", f"{run}/{c}", dirs_exist_ok=True)
            print("reused", c, "from reference run R2")

def show(pattern, width=900):
    for path in sorted(glob.glob(pattern)):
        print(path)
        display(Image(path, width=width))

def result_row(run, condition):
    """One condition's row from intervention_results.csv, written by guidance_intervene_eval.py --stats."""
    if not os.path.exists(f"{run}/intervention_results.csv"):
        return None
    for r in csv.DictReader(open(f"{run}/intervention_results.csv")):
        if r["condition"] == condition:
            return {k: (v if k == "condition" else float(v)) for k, v in r.items()}

def print_table(rows):
    """rows: list of (label, result_row). Recovery: 0 = like w = 4, 1 = like w = 1. Retention: 0 = like w = 1, 1 = like w = 4."""
    print(f"{'':<14}{'layout':>8}{'DINOv2':>8}{'CLIP':>8}{'retention':>11}   (diversity recovered | recognizability kept)")
    for label, r in rows:
        print(f"{label:<14}{r['recovery_V_low']:>8.2f}{r['recovery_dino']:>8.2f}{r['recovery_clip']:>8.2f}{r['retention']:>11.2f}")

def tradeoff_plot(points, title, path, line=False):
    """points: list of (label, result_row, style). Diversity recovered vs recognizability kept, 95% bootstrap intervals."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
    for ax, key, name in zip(axes, ("V_low", "dino", "clip"), ("Layout", "DINOv2", "CLIP")):
        if line:
            ax.plot([r["retention"] for _, r, _ in points], [r[f"recovery_{key}"] for _, r, _ in points], "-", color="#9bbbe6", zorder=0)
        for label, r, style in points:
            x, y = r["retention"], r[f"recovery_{key}"]
            ax.errorbar(x, y, xerr=[[x - r["retention_low"]], [r["retention_high"] - x]],
                        yerr=[[y - r[f"recovery_{key}_low"]], [r[f"recovery_{key}_high"] - y]], capsize=3, **style)
            ax.annotate(label, (x, y), xytext=(6, 6), textcoords="offset points", fontsize=9)
        ax.set_title(f"{name} diversity recovered")
        ax.set_xlabel("recognizability kept (1 = like w = 4)")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("diversity recovered (1 = like w = 1)")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()

BLUE = dict(fmt="o", color="#2a78d6", ecolor="#9bbbe6")
ORANGE = dict(fmt="*", ms=14, color="#d6572a", ecolor="#f0b49c")
GREEN = dict(fmt="D", color="#2a9d5c", ecolor="#a3d9b9")
print("ready")

# Experiments

Both experiments change guidance only during the early part of sampling and leave it at full w = 4 after. Both use
held-out seeds 16–31 and the same cost as plain guidance (2 network passes per step), and both are scored the same way:

- **Diversity recovered** (0 = as uniform as w = 4, 1 = as varied as w = 1) on three measures: *layout* (the coarse
  pattern of the image), and *DINOv2* and *CLIP* (how different the images look to two image-recognition models).
- **Recognizability kept** (0 = like w = 1, 1 = like w = 4): an ImageNet classifier's confidence in the right class.

That classifier's confidence is near 100% in most conditions and misses problems like washed-out color, so always look
at the sample images too.

## Experiment 1: Guidance interval — when should guidance start?

**Idea.** Run with no extra push (w = 1) at the start, then switch to full w = 4 partway through. The artists sketch
their own layouts first, and the coach only arrives to improve details. This is the published "guidance interval"
idea (Kynkäänniemi et al., 2024).

**Question.** How does the start time trade variety against recognizability? The original study tried only t = 0.3
(layout came back almost fully, but only 84% of recognizability was kept). Does an earlier start keep more
recognizability, and does a later one add more variety? Does any start time match score damping?

**What it runs.** For each start time t = 0.1, 0.2, 0.3, 0.4 and 0.5: w = 1 before it, w = 4 after it. The w = 1 and
w = 4 baselines don't depend on the start time, so they are sampled once and shared (or reused from reference run R2).

**Time:** ~10–15 minutes per start time, plus ~30 minutes for the baselines unless reference run R2 has been done.

In [ ]:
STARTS = [0.1, 0.2, 0.3, 0.4, 0.5]

# Baselines, sampled once: from reference run R2 if it has them, otherwise here.
BASE = f"{EXP1}/baselines"
os.makedirs(BASE, exist_ok=True)
reuse_from_ref2(BASE, ["w1", "w4"])
!python bench/guidance_intervene.py --tables {T} --weights {W} --out {BASE} --seeds {TEST_SEED_ARG} --conditions w1 w4 --device cuda

# One run per start time: copy the baselines in, then sample scalar_1 with the window ending at that time.
for start in STARTS:
    run = f"{EXP1}/start_{start}"
    os.makedirs(run, exist_ok=True)
    for c in ("w1", "w4"):
        if not have_all(f"{run}/{c}"):
            shutil.copytree(f"{BASE}/{c}", f"{run}/{c}", dirs_exist_ok=True)
    print(f"--- guidance starts at t = {start}")
    !python bench/guidance_intervene.py --tables {T} --weights {W} --out {run} --seeds {TEST_SEED_ARG} --conditions scalar_1 --early-end {start} --device cuda

### Scoring

Decodes the final images, embeds them with DINOv2 and CLIP, scores them with an ImageNet classifier, then computes
recovery and retention. The models download from Hugging Face on first use.

In [ ]:
for start in STARTS:
    !python bench/guidance_intervene_eval.py --run {EXP1}/start_{start} --embed --stats

### Results

Up and to the right is better. Later start times should move points up (more variety) and left (less recognizable).
If a start time reaches the orange star (score damping, shown if reference run R2 or experiment 2 has run), the simple
on/off switch matches it; if every point stays below and left of it, targeting the shared push is doing real work.

In [ ]:
rows = [(f"t = {s}", result_row(f"{EXP1}/start_{s}", "scalar_1")) for s in STARTS]
score = result_row(EXP2, "score_1") or result_row(REF2, "score_1")

print_table(rows + ([("score damping", score)] if score else []))
points = [(label, r, BLUE) for label, r in rows] + ([("score damping", score, ORANGE)] if score else [])
tradeoff_plot(points, "Experiment 1: when guidance starts (seeds 16-31, 95% bootstrap intervals)", f"{EXP1}/start_time_sweep.png", line=True)

Samples for one seed at each start time (rows are the 8 classes in a 4 × 2 grid). Look for broken or washed-out
objects at late start times, which the recognizability score can miss.

In [ ]:
SEED = 23
show(f"{EXP1}/start_*/samples/scalar_1_seed_{SEED}.png", width=500)

## Experiment 2: Shuffled push — is the problem the push, or that everyone gets the same one?

**Idea.** Score damping finds each class's *shared push*, the part of early guidance that is the same for every sample
of the class (3 directions per class), and deletes its extra strength. The shuffled push keeps that extra strength but
turns it to a different direction for each sample, within the same 3 directions. Every artist still gets a firm
layout instruction, just not the same one.

```
score damping:   f = f_cond + 3·(Δ − λ·shared)
shuffled push:   f = f_cond + 3·(Δ − λ·shared + λ·rotate(shared))
```

`rotate` is a random rotation of the 3 shared directions, drawn once per seed and class and kept the same at every
step (so the push points consistently one way). It changes the push's direction but not its size, and leaves the rest
of guidance alone.

**Question and how to read the answer.**

| If shuffled push... | It means... |
|---|---|
| recovers variety **and** keeps more recognizability than score damping | early guidance is useful; the problem is only that every sample gets the same push |
| recovers variety but loses recognizability | the shared push's direction *is* the class signal; only removing or weakening it works |
| recovers little variety | a strong push in any of these directions still makes samples alike |

**Known risk.** A random rotation can turn the push close to its opposite (an "anti-class" push), which may hurt some
samples. Half strength (λ = 0.5) limits this, which is why it is included.

**What it runs.** On seeds 16–31: `w1`, `w4`, `score_1` (score damping, for comparison), `shuffle_0.5` and
`shuffle_1`. Conditions already done in reference run R2 are reused.

**Time:** ~15 minutes per condition, so ~45–75 minutes depending on reuse.

In [ ]:
# Check the rotation code before sampling: rotations must be orthogonal with determinant +1, reproducible per seed.
from guidance_intervene import random_rotations
R = random_rotations(16, 8)
assert torch.allclose(R @ R.transpose(1, 2), torch.eye(3).expand(8, 3, 3), atol=1e-5), "rotations are not orthogonal"
assert torch.allclose(torch.det(R), torch.ones(8), atol=1e-5), "a rotation has determinant -1"
assert torch.equal(R, random_rotations(16, 8)) and not torch.equal(R, random_rotations(17, 8)), "rotations are not seeded"
print("rotation check passed")

In [ ]:
EXP2_CONDITIONS = ["w1", "w4", "score_1", "shuffle_0.5", "shuffle_1"]
EXP2_ARG = " ".join(EXP2_CONDITIONS)
os.makedirs(EXP2, exist_ok=True)
reuse_from_ref2(EXP2, ["w1", "w4", "score_1"])
!python bench/guidance_intervene.py --tables {T} --weights {W} --out {EXP2} --seeds {TEST_SEED_ARG} --conditions {EXP2_ARG} --device cuda

### Scoring

In [ ]:
!python bench/guidance_intervene_eval.py --run {EXP2} --embed --stats

### Results

Compare the green diamonds (shuffled push) with the orange star (score damping): up and to the right is better.

In [ ]:
rows = [(c, result_row(EXP2, c)) for c in EXP2_CONDITIONS]
print_table(rows)
style = {"w1": BLUE, "w4": BLUE, "score_1": ORANGE, "shuffle_0.5": GREEN, "shuffle_1": GREEN}
tradeoff_plot([(c, r, style[c]) for c, r in rows], "Experiment 2: shuffled push vs score damping (seeds 16-31, 95% bootstrap intervals)",
              f"{EXP2}/shuffled_push.png")

Samples for one seed under each condition (rows are the 8 classes in a 4 × 2 grid). With the shuffled push, layouts
should differ from w = 4's; look for odd or off-class images, which would point to the "anti-class" risk.

In [ ]:
SEED = 23
for c in EXP2_CONDITIONS:
    print(c)
    display(Image(f"{EXP2}/samples/{c}_seed_{SEED}.png", width=500))

## Experiment 3: Limited turn — does capping the angle remove the recoloring?

**Background.** In experiment 2 the shuffled push recovered more variety than score damping at the same
recognizability, but it strongly recolored 25 of 128 images (score damping: 2), some visibly wrong: an underexposed
dog, a pink geyser. Under a fully random rotation the push turns by 90° on average and about half of all pushes end up
partly pointing backwards, which is the obvious suspect.

**Idea.** `turn_A` turns each sample's push by exactly A degrees about a random axis of the class's 3 shared
directions. That moves the push by **at most** A degrees, so below 90° it never points backwards.

| Condition | Largest possible turn | Average turn |
|---|---|---|
| `turn_30` | 30° | ~23° |
| `turn_60` | 60° | ~47° |
| `turn_90` | 90° | ~69° |
| `turn_120` | 120° | ~85° |
| `shuffle_1` (experiment 2) | 180° | 90° |

**Question.** Is there an angle that keeps most of the shuffled push's extra variety but recolors about as little as
score damping? If the recoloring shrinks with the angle while variety holds up, the backward-pointing pushes were the
problem.

**What it runs.** The four `turn` conditions on seeds 16–31. `w1`, `w4`, `score_1` and `shuffle_1` are copied from
experiment 2, which must have run first.

**Time:** ~12 minutes per condition, about 50 minutes in all.

In [ ]:
# Check the fixed-angle rotations before sampling: proper rotations that never turn a vector further than the angle.
from guidance_intervene import angle_rotations
for deg in (30, 60, 90, 120):
    R = angle_rotations(16, 8, deg)
    v = torch.randn(8, 3, dtype=torch.float32)
    turn = torch.rad2deg(torch.arccos(((R @ v[:, :, None])[:, :, 0] * v).sum(1) / (v * v).sum(1)).clamp(-1, 1))
    assert torch.allclose(R @ R.transpose(1, 2), torch.eye(3).expand(8, 3, 3), atol=1e-5), "not orthogonal"
    assert torch.allclose(torch.det(R), torch.ones(8), atol=1e-5), "determinant is not +1"
    assert (turn <= deg + 1e-3).all(), f"a {deg} degree rotation turned a vector by {turn.max():.1f} degrees"
print("angle check passed")

In [ ]:
EXP3_NEW = ["turn_30", "turn_60", "turn_90", "turn_120"]
EXP3_CONDITIONS = ["w1", "w4", "score_1", "shuffle_1"] + EXP3_NEW
EXP3_ARG = " ".join(EXP3_NEW)

assert all(have_all(f"{EXP2}/{c}") for c in ("w1", "w4", "score_1", "shuffle_1")), "run experiment 2 first"
os.makedirs(EXP3, exist_ok=True)
for c in ("w1", "w4", "score_1", "shuffle_1"):
    if not have_all(f"{EXP3}/{c}"):
        shutil.copytree(f"{EXP2}/{c}", f"{EXP3}/{c}", dirs_exist_ok=True)
!python bench/guidance_intervene.py --tables {T} --weights {W} --out {EXP3} --seeds {TEST_SEED_ARG} --conditions {EXP3_ARG} --device cuda

### Scoring

In [ ]:
!python bench/guidance_intervene_eval.py --run {EXP3} --embed --stats

### Results

Variety and recognizability as before, plus recoloring: how many of the 128 images changed their overall color
strongly compared with the same seed and class at w = 4 (score damping: 2, shuffled push: 25 in experiment 2).
The best angle keeps the shuffled push's variety with a recoloring count near score damping's.

In [ ]:
from guidance_new_figures import color_stats
%matplotlib inline
rows = [(c, result_row(EXP3, c)) for c in EXP3_CONDITIONS]
print_table(rows)

samples = {c: __import__("pathlib").Path(EXP3) / "samples" for c in EXP3_CONDITIONS}
color_rows, _ = color_stats(samples, EXP3_CONDITIONS)
print(f"\n{'condition':<14}{'strongly recoloured':>21}{'mean colour change':>20}{'saturation':>12}")
for r in color_rows:
    print(f"{r['condition']:<14}{r['strongly_recoloured']:>15} / 128{r['mean_shift']:>20.3f}{r['mean_saturation']:>12.3f}")

style = {"w1": BLUE, "w4": BLUE, "score_1": ORANGE, "shuffle_1": GREEN}
style.update({c: dict(fmt="D", color="#7a4bc2", ecolor="#c8b3ea") for c in EXP3_NEW})
tradeoff_plot([(c, r, style[c]) for c, r in rows if c not in ("w1", "w4")],
              "Experiment 3: limited turn vs full shuffle and score damping (seeds 16-31, 95% bootstrap intervals)", f"{EXP3}/limited_turn.png")

Samples for the seeds where the shuffled push went most visibly wrong (16: dark retriever; 19: pink geyser and a
second red panda). Compare each turn angle with the full shuffle.

In [ ]:
for seed in (16, 19):
    for c in ["w4", "score_1", "shuffle_1"] + EXP3_NEW:
        print(f"seed {seed}: {c}")
        display(Image(f"{EXP3}/samples/{c}_seed_{seed}.png", width=500))

# Reference runs (optional)

These reproduce results from the original study (RESULTS.md) on Colab. They are not needed for the experiments above,
but if R2 has run, both experiments reuse its baselines and compare against its score damping result.

## R1. The guidance vector over time

**Question.** How large is the guidance vector Δ = f_cond − f_uncond at each sampling step, and does its direction
stay stable from one step to the next? (RESULTS.md part 4.)

**What it runs.** 8 seeds × 8 classes at w = 4, storing Δ at all 49 steps. Then it checks the run: seed 0 is rerun to
confirm it reproduces, each Euler step is rebuilt from the logged values, and a few steps are recomputed on the CPU.
(The README's version also compares against the official XL/2 model; that needs a second 2.7 GB download and is
skipped here.)

**Time:** ~5–10 minutes. It is also the quickest way to confirm the setup works.

In [ ]:
!python bench/guidance_delta.py --model SiT-L/2 --weights {W} --weights-source "{SRC}" --out {REF1} --seeds 0 1 2 3 4 5 6 7 --steps 50 --cfg 4.0 --device cuda
!python bench/guidance_verify.py --run {REF1} --seed 0
!python bench/guidance_plot.py --run {REF1}
show(f"{REF1}/guidance_delta.png")

## R2. The original interventions

**Question.** Which change to guidance during t < 0.3 brings back variety while keeping recognizability?
(RESULTS.md part 10.)

**What it runs.** 13 conditions on held-out seeds 16–31:

| Condition | What it changes for t < 0.3 |
|---|---|
| `w1`, `w4` | Baselines: plain w = 1 and plain w = 4 throughout. |
| `scalar_0.5`, `scalar_1` | Lowers w for every direction equally (the control). λ = 1 means w = 1 early. |
| `score_0.5`, `score_1` | Removes extra guidance only along each class's 3 shared directions. |
| `freq_0.5`, `freq_1` | Removes the coarse (low-frequency) part of extra guidance. |
| `cond_0.5`, `cond_1` | Weakens the class label fed into the network. |
| `feat_0.5`, `feat_1` | Removes the shared class part of activations in blocks 13–23. |
| `wrongclass_1` | Score damping with another class's directions (the control for targeting). |

**Time:** ~2–4 hours for all 13. For a ~1 hour run, set `CONDITIONS` to a subset such as
`["w1", "w4", "scalar_1", "score_1", "cond_0.5"]` (`w1` and `w4` are always needed). The summary figure needs all 13,
so it is skipped for a subset.

In [ ]:
ALL_CONDITIONS = ["w1", "w4", "scalar_0.5", "scalar_1", "score_0.5", "score_1", "freq_0.5", "freq_1",
                  "cond_0.5", "cond_1", "feat_0.5", "feat_1", "wrongclass_1"]
CONDITIONS = ALL_CONDITIONS
assert {"w1", "w4"} <= set(CONDITIONS)
COND_ARG = " ".join(CONDITIONS)

!python bench/guidance_intervene.py --tables {T} --weights {W} --out {REF2} --seeds {TEST_SEED_ARG} --conditions {COND_ARG} --device cuda
FIG = "--figures" if set(CONDITIONS) == set(ALL_CONDITIONS) else ""
!python bench/guidance_intervene_eval.py --run {REF2} --embed --stats {FIG}

In [ ]:
# For reference, the original run found score_1 recovers all lost layout diversity while keeping 97% of recognizability.
print_table([(c, result_row(REF2, c)) for c in CONDITIONS])
show(f"{REF2}/figures/*.png")

## R3. Full logging of w = 4 vs w = 1 (off by default)

**Question.** Step by step, what does guidance do? How large Δ is and which way it points, whether seeds of a class
share a direction, how its frequency content changes, and how much diversity it removes. (RESULTS.md parts 6–7.)

**What it runs.** 16 seeds at w = 4 and w = 1 with full logging, then verification, embedding, statistics and
figures 1–9. About 19 GB, more than free Drive holds, so it writes to the session's local disk and copies only
`analysis/` (tables and figures) to Drive at the end. Set `RUN_REF3 = True` to run it.

In [ ]:
!df -h /content | tail -1
RUN_REF3 = False

if RUN_REF3:
    SEEDS16 = " ".join(map(str, range(16)))
    !python bench/guidance_capture.py --weights {W} --weights-source "{SRC}" --out {REF3} --seeds {SEEDS16} --device cuda
    FIRST_ARG = f"--first-run {REF1}" if os.path.exists(f"{REF1}/manifest.json") else ""  # compares with R1 if it ran
    !python bench/guidance_capture_verify.py --run {REF3} {FIRST_ARG}
    !python bench/guidance_embed.py --run {REF3} --device cuda
    !python bench/guidance_analysis.py --run {REF3}
    !python bench/guidance_figures.py --run {REF3} --only 1 2 3 4 5 6 7 8 9
    if USE_DRIVE:
        !mkdir -p {OUT}/ref3_16seeds && cp -r {REF3}/analysis {REF3}/manifest.json {OUT}/ref3_16seeds/
    show(f"{REF3}/analysis/figures/*.png")